# Day 11: Optimal Control — From HJB to Portfolio Weights

*Alpha Flow Research · HongJin HE · July 2026*

---

## The Controller C Problem

Given the Nash equilibrium mean-field $\mu^*$ from the Game Module G, the Controller C solves a **stochastic optimal control problem** for our portfolio.

**State**: wealth $W_t$, latent market state $z_t$

**Control**: portfolio weights $w_t \in \Delta^n$ (simplex constraint)

**Value function**:
$$V(t, W, z) = \sup_{w \in \Delta^n} \mathbb{E}\left[U(W_T) - \lambda \int_t^T c(w_s) ds \mid W_t = W, z_t = z\right]$$

where $U(W) = \log W$ (Merton's log utility) and $c(w) = \kappa \|w - w_\text{prev}\|_1$ (transaction cost).

## The HJB Equation

By the stochastic maximum principle, $V$ satisfies the **Hamilton-Jacobi-Bellman PDE**:

$$\partial_t V + \sup_{w \in \Delta^n} \left[\nabla_W V \cdot W(w^\top \mu_t) + \nabla_z V \cdot b(z,\mu^*) + \frac{1}{2} W^2 \nabla_{WW} V \cdot w^\top \Sigma w\right] - \lambda c(w) = 0$$

Terminal condition: $V(T, W, z) = \log W$

The optimal weights satisfy: $w^* = \arg\sup_w [\cdots]$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

np.random.seed(2026)

# ── Merton's closed-form solution (no transaction costs, GBM) ────────────
# w* = Σ⁻¹ μ / γ  (where γ is risk aversion, = 1 for log utility)

n_assets = 5
# Simulate expected returns and covariance
mu_assets = np.array([0.10, 0.08, 0.12, 0.06, 0.14]) / 252  # daily expected returns

# Covariance matrix (factor structure)
factor_loadings = np.random.randn(n_assets, 3)  # 3 factors
idio_vol = np.diag(np.random.uniform(0.1, 0.3, n_assets)**2 / 252)
Sigma = factor_loadings @ factor_loadings.T * 0.01/252 + idio_vol

# Merton optimal weights (unconstrained)
Sigma_inv = np.linalg.inv(Sigma)
gamma = 1.0  # log utility → γ = 1
w_merton = Sigma_inv @ mu_assets / gamma
print('Merton unconstrained weights (can go negative/>1):')
print(f'  {w_merton}')
print(f'  Sum: {w_merton.sum():.3f}')

# ── Constrained optimization (simplex: w ≥ 0, Σw = 1) ───────────────────
def neg_sharpe(w, mu, Sigma, gamma=1.0):
    """Maximize expected log utility: μ'w - γ/2 w'Σw"""
    return -(mu @ w - gamma/2 * w @ Sigma @ w)

constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1.0}]
bounds = [(0.0, 1.0)] * n_assets
w0 = np.ones(n_assets) / n_assets

result = minimize(neg_sharpe, w0, args=(mu_assets, Sigma, gamma),
                  constraints=constraints, bounds=bounds, method='SLSQP')
w_constrained = result.x

print('\nConstrained (simplex) weights:')
for i, w in enumerate(w_constrained):
    print(f'  Asset {i}: {w:.3f}')
print(f'  Sum: {w_constrained.sum():.6f}')

# Portfolio statistics
ann_factor = 252
w_test = w_constrained
port_return = (mu_assets @ w_test) * ann_factor
port_vol = np.sqrt(w_test @ Sigma @ w_test * ann_factor)
sharpe = port_return / port_vol
print(f'\nPortfolio: Return={port_return:.1%}, Vol={port_vol:.1%}, Sharpe={sharpe:.2f}')

In [ ]:
# ── Efficient frontier with MFG drift vs constant drift ──────────────────
n_portfolios = 5000
target_returns = np.linspace(mu_assets.min(), mu_assets.max(), 50) * ann_factor

frontier_vols, frontier_rets = [], []
for target_ret in target_returns:
    constraints_ef = [
        {'type': 'eq', 'fun': lambda w: w.sum() - 1.0},
        {'type': 'eq', 'fun': lambda w, r=target_ret: (mu_assets @ w) * ann_factor - r}
    ]
    res = minimize(lambda w: w @ Sigma @ w * ann_factor, w0,
                  constraints=constraints_ef, bounds=bounds, method='SLSQP')
    if res.success:
        frontier_vols.append(np.sqrt(res.fun))
        frontier_rets.append(target_ret)

# Random portfolios
rand_weights = np.random.dirichlet(np.ones(n_assets), n_portfolios)
rand_rets = (rand_weights @ mu_assets) * ann_factor
rand_vols = np.array([np.sqrt(w @ Sigma @ w * ann_factor) for w in rand_weights])

fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(rand_vols, rand_rets, c=rand_rets/rand_vols, cmap='RdYlGn', 
               s=5, alpha=0.5, zorder=1)
plt.colorbar(sc, ax=ax, label='Sharpe ratio')
ax.plot(frontier_vols, frontier_rets, 'k-', lw=2.5, label='Efficient frontier', zorder=3)
ax.scatter([port_vol], [port_return], s=200, color='red', zorder=5, 
          label=f'MFG-optimal (SR={sharpe:.2f})', marker='*')
ax.scatter([np.sqrt(w_merton @ Sigma @ w_merton * ann_factor)], 
          [mu_assets @ w_merton * ann_factor],
          s=200, color='blue', zorder=5, marker='^', label='Unconstrained Merton')
ax.set_xlabel('Annualised Volatility')
ax.set_ylabel('Annualised Return')
ax.set_title('Efficient Frontier: Controller C Output')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/efficient_frontier.png', dpi=150, bbox_inches='tight')
plt.show()

## How MFG Changes the Optimal Portfolio

In the standard Merton framework, $\mu$ and $\Sigma$ are **exogenous** — fixed parameters from the data.

In MicroWorld, the drift $\mu_t = \mu(z_t, \mu^*)$ **depends on the mean-field $\mu^*$** — the aggregate actions of all other agents:

$$\mu_t = r_f + \underbrace{\beta(z_t)}_{\text{market beta}} + \underbrace{\alpha(z_t, \mu^*_t)}_{\text{MFG alpha, crowding-adjusted}}$$

The MFG alpha term **automatically adjusts for crowding**: if too many agents are long a factor, $\alpha$ decreases, reducing the optimal allocation. This prevents the capital crowding that kills factor returns (see Day 1).

**Implementation**: see `controller/portfolio.py` for the full MFG-adjusted weight computation.